# Draftly — Case-Law Extraction: Method Validation (pre-registered)

**Purpose.** Before spending credits on the full ~3,179 no-headnote run, *show*
the extraction method works and *compare* it to alternatives — an ablation, not an
assertion. This notebook is a **presentation layer** over frozen artifacts produced
by `scripts/case-law-information-extraction/ablation.py`; it makes **no API calls**,
so every number here is reproducible from the saved responses.

**Honesty caveats (read first):**
- **Quote grounding is necessary, not sufficient.** A verbatim quote can be obiter,
  a party's submission, or quoted precedent, and may not support the paraphrase.
  Grounding is checked two independent ways (char-based gate + a token-based
  second opinion); legal *correctness* is a separate question.
- **The LLM-judge is a PROXY, not authoritative.** The `lawyer-labels.csv` pack is
  for a human lawyer; the GO decision rests on those labels.
- **Hosted models are not bit-reproducible even at temperature 0** — which is why
  raw responses, sample, model IDs, prompt hashes, and the code commit are frozen.

## Setup — load the frozen run artifacts

In [ ]:
import json, sys
from pathlib import Path
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "evaluation").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
RUN = ROOT / "evaluation" / "runs" / "caselaw-ablation-v1"
cfg = json.loads((RUN / "config.json").read_text(encoding="utf-8"))
metrics = json.loads((RUN / "metrics.json").read_text(encoding="utf-8"))
preds = pd.read_csv(RUN / "predictions.csv")
pd.set_option("display.max_colwidth", 150)
print("run:", cfg["run_id"], "| commit:", cfg["code_commit"], "| seed:", cfg["seed"])
print("artifacts:", sorted(p.name for p in RUN.glob("*")))

## Stage 1 — Samples & stratification

The population is CommonLII cases cached `no-headnote` (Track A found no `Held:`
block): **3,179 cases, ~93% Court of Appeal, ~68% pre-1950, only 22 post-2000.**
The **ablation** sample deliberately over-samples *at-risk* cases (ruling only in
the omitted middle) and LKSC so arm differences are detectable; the **holdout** is
proportional/representative and is what the precision estimate uses.

In [ ]:
abl_s = pd.read_csv(RUN / "sample_ablation.csv")
hold_s = pd.read_csv(RUN / "sample_holdout.csv")
def compo(df, name):
    print(f"{name}: n={len(df)}")
    for k in ["court", "at_risk", "year_bucket", "length_bucket"]:
        print("  ", k, df[k].value_counts().to_dict())
compo(abl_s, "ablation (coverage-oriented)")
compo(hold_s, "holdout (representative)")

## Stage 2 — Method ablation (4 arms)

| Arm | Model | Window |
|---|---|---|
| A | 8B | naive head+tail |
| B | 8B | cue-aware (shipped) |
| C | 8B | full text |
| D | 70B | full text |

`accept_rate` = produced a rule that passed the strict quote gate.
`independent_grounding_rate` = of accepted rules, the fraction whose quote also
passes the *token-based* second-opinion check (should be ~1.0).
`cue_in_window_rate` = ruling cue present in the model's window (a *necessary*
condition, not proof the holding was extracted).

In [ ]:
arms = metrics["arms"]
row = []
for a in sorted(arms):
    m = arms[a]
    row.append({"arm": a, "model": m["spec"].get("model","").split("/")[-1],
                "window": m["spec"].get("window"), "n": m["n"],
                "accept": m["accept_rate"], "verbatim": m["verbatim_rate"],
                "indep_grounded": m["independent_grounding_rate"],
                "cue_in_window": m["cue_in_window_rate"],
                "at_risk_cue_recovered": m["at_risk_cue_recovered"],
                "abstain": m["abstained"], "reject": m["rejected"], "err": m["llm_errors"]})
display(pd.DataFrame(row).set_index("arm"))
print("frozen best arm:", (metrics.get("best_method") or {}).get("arm"))

### The specific windowing claim: A (naive) vs B (cue-aware) on *at-risk* cases

This is the case the whole windowing change rests on — where the ruling sits in the
omitted middle. If B > A here (higher cue-in-window and accept), the cue-aware
window is doing its job.

In [ ]:
ar = preds[(preds.stage=="ablation") & (preds.at_risk==True)]
for a in ["A","B"]:
    s = ar[ar.arm==a]
    acc = (s.status=="ok").sum()
    cue = s.cue_in_window.sum()
    print(f"arm {a}: at_risk n={len(s)} | cue_in_window={cue} | accepted={acc}")

## Stage 3 — Holdout (frozen best arm) + LLM-judge proxy

The best arm is re-run on the fresh, disjoint, representative 100-case holdout.
Every *accepted* rule is scored by a 70B judge on six dimensions. **This is a proxy**
— the authoritative labels live in `lawyer-labels.csv`. Wilson 95% CI lower bounds
are shown because n is small.

In [ ]:
h = metrics["holdout"]
print(f"holdout n={h['n']} | accept_rate={h['accept_rate']} | indep_grounding={h['independent_grounding_rate']}")
dims=["statement_correct","quote_supports","is_ratio","qualifications_ok","statute_correct","usable"]
cj = h.get("judge_claude", {}); jl = h.get("judge_proxy", {})
def jtable(src, label, n):
    print(f"\n{label} (n={n}, PROXY - non-authoritative):")
    rows=[{"dimension":d, "rate":src.get(d,{}).get("rate"), "ci95_lower":src.get(d,{}).get("ci95_lower"),
           "k":src.get(d,{}).get("k"), "n":src.get(d,{}).get("n")} for d in dims]
    display(pd.DataFrame(rows).set_index("dimension"))
if cj.get("n"): jtable(cj, "Claude-Opus judge (STRONGER proxy)", cj["n"])
if jl.get("n_judged"): jtable(jl, "llama-3.3-70b judge (weaker proxy, may be partial)", jl["n_judged"])
print("\nprimary proxy used for the provisional verdict:", h.get("provisional_judge_source"))
print(h["judge_note"])

### Qualitative sample — accepted holdout rules (for human eyeballing)

In [ ]:
acc = preds[(preds.stage=="holdout") & (preds.status=="ok")].copy()
show = acc[["citation","statement","supporting_quote","statute_section"]].head(15)
show["supporting_quote"] = show["supporting_quote"].str.slice(0,140)
show["statement"] = show["statement"].str.slice(0,140)
display(show.reset_index(drop=True))
print(f"total accepted holdout rules: {len(acc)}  (full pack in lawyer-labels.csv)")

## Verdict — provisional (proxy-based) vs the real gate

The GO criteria were **pre-registered** in `config.json` before any results existed.
The verdict below is **provisional**, computed from the LLM-judge proxy. **GO is only
authorised by the human lawyer labels** in `lawyer-labels.csv`.

In [ ]:
gc = cfg["go_criteria"]
pv = metrics["provisional_verdict"]
print("PRE-REGISTERED GO CRITERIA:")
for k,v in gc.items(): print(f"  {k}: {v}")
print("\nPROVISIONAL CHECKS (LLM-judge proxy):")
for k,v in pv["checks"].items(): print(f"  {'PASS' if v else 'FAIL'}  {k}")
print(f"\nprovisional_all_pass = {pv['all_pass']}  (authoritative={pv['authoritative']})")
print(pv["note"])
cr = metrics["credits"]
print(f"\nCREDITS (logical model calls, counted from raw-responses.jsonl): total={cr['total_logical_calls']}")
for m,n in cr["logical_calls_by_model"].items(): print(f"   {m}: {n}")
print(cr["note"])

## Next step — finalize GO with a human lawyer

`evaluation/runs/caselaw-ablation-v1/lawyer-labels.csv` contains every accepted
holdout rule with blank label columns. A lawyer fills y/n for statement_correct,
quote_supports, is_ratio, qualifications_ok, statute_correct, usable. Re-compute the
same metrics on those labels; **GO only if all pre-registered criteria hold on the
human labels.** Only then run the full ~3,179-case Phase 1.

Everything here is reproducible: `raw-responses.jsonl` (frozen model outputs),
`predictions.csv`, `sample_*.csv`, `metrics.json`, `config.json` (commit-pinned).